# Sentiment Labeling (IndoBERT)


In [54]:
import pandas as pd
import numpy as np
import re
import json
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import emoji as emoji_lib

print('Torch version   :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU              :', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU tidak terdeteksi. Di Colab: Runtime -> Change runtime type -> GPU (T4).')

Torch version   : 2.11.0+cpu
CUDA available  : False


## Load Data Bersih (Output Tahap 4) & Siapkan Teks untuk BERT

In [55]:
df = pd.read_parquet('../data/interim/clean_comments_kopdes.parquet')
print(f'Baris input: {len(df):,}')

def case_folding(s): return s.lower()
def remove_url(s): return re.sub(r'https?://\S+|www\.\S+', ' ', s)
def remove_mention(s): return re.sub(r'@\w+', ' ', s)
def remove_hashtag(s): return re.sub(r'#\w+', ' ', s)
def remove_system_tags(s): return re.sub(r'\[sticker\]', ' ', s, flags=re.IGNORECASE)
def remove_emoji(s): return emoji_lib.replace_emoji(s, replace=' ')
def normalize_elongation(s): return re.sub(r'(.)\1{2,}', r'\1\1', s)
def normalize_space(s): return re.sub(r'\s+', ' ', s).strip()

df['text_for_bert'] = (
    df['comment'].astype(str)
    .apply(case_folding).apply(remove_url).apply(remove_mention)
    .apply(remove_hashtag).apply(remove_system_tags).apply(remove_emoji)
    .apply(normalize_elongation).apply(normalize_space)
)

n_before = len(df)
df = df[df['text_for_bert'].str.strip() != ''].reset_index(drop=True)
print(f'Drop residu kosong pasca light-cleaning: {n_before:,} -> {len(df):,}')
df[['comment', 'text_for_bert']].head(5)

Baris input: 120,707
Drop residu kosong pasca light-cleaning: 120,707 -> 119,565


,comment,text_for_bert
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya 2 m ?
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m
3,[Sticker] dibilang Makasih Mas udah 2M itu,dibilang makasih mas udah 2m itu
4,"Kalo niat membantu UMKM, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu...","kalo niat membantu umkm, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu..."


## Load Model IndoBERT Sentiment

In [56]:
MODEL_NAME = 'mdhugol/indonesia-bert-sentiment-classification'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print('Raw id2label dari model config:')
print(model.config.id2label)

def normalize_label(raw_label: str) -> str:
    s = raw_label.lower()
    if 'pos' in s:
        return 'positive'
    elif 'neg' in s:
        return 'negative'
    elif 'neu' in s:
        return 'neutral'
    else:
        mapping_fallback = {'label_0': 'positive', 'label_1': 'neutral', 'label_2': 'negative'}
        return mapping_fallback.get(s, s)

id2label_normalized = {k: normalize_label(v) for k, v in model.config.id2label.items()}
print()
print('Mapping ternormalisasi:', id2label_normalized)
print()
print('>>> VERIFIKASI MANUAL: cocokkan mapping di atas dengan model card di halaman Hugging Face')
print('>>> sebelum lanjut ke inferensi penuh (jalankan cell contoh kalimat di bawah).')

config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Raw id2label dari model config:
{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}

Mapping ternormalisasi: {0: 'positive', 1: 'neutral', 2: 'negative'}

>>> VERIFIKASI MANUAL: cocokkan mapping di atas dengan model card di halaman Hugging Face
>>> sebelum lanjut ke inferensi penuh (jalankan cell contoh kalimat di bawah).


## Uji Cepat pada Contoh Kalimat (Sanity Check Sebelum Full Run)

In [57]:
device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=128,
    top_k=None
)

test_sentences = [
    'koperasi ini sangat membantu masyarakat desa, terima kasih pemerintah',   # ekspektasi: positive
    'saya tidak tahu apakah program ini akan berhasil atau tidak',              # ekspektasi: neutral
    'program ini gagal total dan hanya buang-buang anggaran negara',           # ekspektasi: negative
]

for s in test_sentences:
    result = sentiment_pipe(s)[0]
    best = max(result, key=lambda x: x['score'])
    label_norm = id2label_normalized[model.config.label2id[best['label']]] if False else normalize_label(best['label'])
    print(f'[{label_norm:9s}] (conf={best["score"]:.3f})  {s}')

[positive ] (conf=0.983)  koperasi ini sangat membantu masyarakat desa, terima kasih pemerintah
[negative ] (conf=0.547)  saya tidak tahu apakah program ini akan berhasil atau tidak
[negative ] (conf=0.997)  program ini gagal total dan hanya buang-buang anggaran negara


## Inferensi Batch pada Seluruh Dataset (dengan Checkpointing)


In [58]:
BATCH_SIZE = 128
CHECKPOINT_PATH = '../data/interim/labeling_checkpoint.parquet'
CHECKPOINT_EVERY_N_BATCHES = 50

texts = df['text_for_bert'].tolist()
n_total = len(texts)

if os.path.exists(CHECKPOINT_PATH):
    ckpt = pd.read_parquet(CHECKPOINT_PATH)
    start_idx = len(ckpt)
    all_labels = ckpt['sentiment_label'].tolist()
    all_scores = ckpt['sentiment_confidence'].tolist()
    print(f'Resume dari checkpoint: {start_idx:,}/{n_total:,} baris sudah diproses sebelumnya.')
else:
    start_idx = 0
    all_labels, all_scores = [], []
    print('Tidak ada checkpoint, mulai dari awal.')

t0 = time.time()
for batch_start in range(start_idx, n_total, BATCH_SIZE):
    batch_texts = texts[batch_start:batch_start + BATCH_SIZE]
    results = sentiment_pipe(batch_texts)

    for r in results:
        best = max(r, key=lambda x: x['score'])
        all_labels.append(normalize_label(best['label']))
        all_scores.append(round(best['score'], 4))

    n_done = batch_start + len(batch_texts)
    batch_num = (batch_start - start_idx) // BATCH_SIZE + 1

    if batch_num % CHECKPOINT_EVERY_N_BATCHES == 0 or n_done >= n_total:
        elapsed = time.time() - t0
        rate = (n_done - start_idx) / elapsed if elapsed > 0 else 0
        remaining = (n_total - n_done) / rate if rate > 0 else 0
        print(f'{n_done:,}/{n_total:,} baris ({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining, {rate:.1f} baris/s)')

        ckpt_df = df.iloc[:n_done].copy()
        ckpt_df['sentiment_label'] = all_labels
        ckpt_df['sentiment_confidence'] = all_scores
        ckpt_df.to_parquet(CHECKPOINT_PATH, index=False)

print()
print(f'INFERENSI SELESAI: {len(all_labels):,} baris berhasil dilabeli dalam {time.time()-t0:.0f}s')

Tidak ada checkpoint, mulai dari awal.
6,400/119,565 baris (860s elapsed, ~15198s remaining, 7.4 baris/s)
12,800/119,565 baris (1577s elapsed, ~13151s remaining, 8.1 baris/s)
19,200/119,565 baris (2223s elapsed, ~11620s remaining, 8.6 baris/s)
25,600/119,565 baris (2847s elapsed, ~10448s remaining, 9.0 baris/s)
32,000/119,565 baris (3538s elapsed, ~9680s remaining, 9.0 baris/s)
38,400/119,565 baris (4290s elapsed, ~9067s remaining, 9.0 baris/s)
44,800/119,565 baris (4968s elapsed, ~8291s remaining, 9.0 baris/s)
51,200/119,565 baris (5591s elapsed, ~7465s remaining, 9.2 baris/s)
57,600/119,565 baris (6314s elapsed, ~6792s remaining, 9.1 baris/s)
64,000/119,565 baris (7074s elapsed, ~6142s remaining, 9.0 baris/s)
70,400/119,565 baris (7810s elapsed, ~5454s remaining, 9.0 baris/s)
76,800/119,565 baris (8562s elapsed, ~4767s remaining, 9.0 baris/s)
83,200/119,565 baris (9260s elapsed, ~4047s remaining, 9.0 baris/s)
89,600/119,565 baris (9945s elapsed, ~3326s remaining, 9.0 baris/s)
96,000/

## Gabungkan Hasil Label & Validasi

In [59]:
df['sentiment_label'] = all_labels
df['sentiment_confidence'] = all_scores

print('=== Distribusi Label Sentimen ===')
print(df['sentiment_label'].value_counts())
print()
print(df['sentiment_label'].value_counts(normalize=True).round(3) * 100, '(%)')

print()
print('=== Statistik Confidence Score ===')
print(df['sentiment_confidence'].describe())

# Validasi
assert df['sentiment_label'].isnull().sum() == 0, 'Ada baris tanpa label!'
assert set(df['sentiment_label'].unique()) <= {'positive', 'neutral', 'negative'}, 'Label tidak standar ditemukan!'
assert len(df) >= 5000
print()
print('VALIDASI PASSED: semua baris berlabel, hanya 3 kelas standar, memenuhi minimum 5.000 baris.')

=== Distribusi Label Sentimen ===
sentiment_label
negative    69362
neutral     27879
positive    22324
Name: count, dtype: int64

sentiment_label
negative    58.0
neutral     23.3
positive    18.7
Name: proportion, dtype: float64 (%)

=== Statistik Confidence Score ===
count    119565.000000
mean          0.888573
std           0.150087
min           0.335500
25%           0.832500
50%           0.967900
75%           0.992900
max           0.998500
Name: sentiment_confidence, dtype: float64

VALIDASI PASSED: semua baris berlabel, hanya 3 kelas standar, memenuhi minimum 5.000 baris.


In [60]:
print('=== Contoh Hasil Labeling (random sample per kelas) ===')
for label in ['positive', 'neutral', 'negative']:
    print(f'--- {label.upper()} ---')
    sample = df[df['sentiment_label'] == label].sample(min(3, (df['sentiment_label']==label).sum()), random_state=1)
    for _, row in sample.iterrows():
        print(f"  (conf={row['sentiment_confidence']:.3f}) {row['comment'][:120]}")
    print()

=== Contoh Hasil Labeling (random sample per kelas) ===
--- POSITIVE ---
  (conf=0.971) katanya membantu umkm
  (conf=0.975) nice info tp aku tetep beli di WARUNG MADURA BUKA 25 JAM TUTUP KALO KIAMAT
  (conf=0.776) mencari keuntungan besar utk kalangan atas

--- NEUTRAL ---
  (conf=0.841) 90% proyek gagal,10% komplit selesai di bangun beserta isinya
  (conf=0.860) kami dr Malaysia ni suka borong barang di indomaret dgn alfamart. Patutlah pg dua kedai ni xde brg. Sbb nk buka koperasi
  (conf=0.984) Ntr pergi log 35k🗿

--- NEGATIVE ---
  (conf=0.994) Ini Koperasi jadi2an, Hnya hasrat memenuhi janji kampanye.
🙏🙏🙏🙏
  (conf=0.997) partai ga bener ngusul kandidat
  (conf=0.991) selamat datang.lah kok ada yang kosong😁



## Simpan Dataset Berlabel

In [61]:
os.makedirs('../data/processed', exist_ok=True)

cols_final = [
    'video_id', 'comment_id', 'parent_comment_id', 'level', 'username', 'nickname',
    'create_time', 'comment', 'text_for_bert', 'sentiment_label', 'sentiment_confidence'
]
df_labeled = df[cols_final].copy()

df_labeled.to_parquet('../data/processed/labeled_comments_kopdes.parquet', index=False)
df_labeled.to_csv('../data/processed/labeled_comments_kopdes.csv', index=False)

labeling_summary = {
    'model_used': MODEL_NAME,
    'n_rows_labeled': int(len(df_labeled)),
    'label_distribution': df_labeled['sentiment_label'].value_counts().to_dict(),
    'label_distribution_pct': (df_labeled['sentiment_label'].value_counts(normalize=True) * 100).round(2).to_dict(),
    'mean_confidence': float(df_labeled['sentiment_confidence'].mean()),
    'median_confidence': float(df_labeled['sentiment_confidence'].median()),
    'low_confidence_rows_below_0.6': int((df_labeled['sentiment_confidence'] < 0.6).sum()),
}
with open('../reports/labeling_summary.json', 'w') as f:
    json.dump(labeling_summary, f, indent=2, default=str)

print(json.dumps(labeling_summary, indent=2, default=str))

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print('\nCheckpoint sementara dihapus (data final sudah tersimpan permanen).')

{
  "model_used": "mdhugol/indonesia-bert-sentiment-classification",
  "n_rows_labeled": 119565,
  "label_distribution": {
    "negative": 69362,
    "neutral": 27879,
    "positive": 22324
  },
  "label_distribution_pct": {
    "negative": 58.01,
    "neutral": 23.32,
    "positive": 18.67
  },
  "mean_confidence": 0.8885728691506714,
  "median_confidence": 0.9679,
  "low_confidence_rows_below_0.6": 10087
}

Checkpoint sementara dihapus (data final sudah tersimpan permanen).


### Kesimpulan Tahap Sentiment Labeling

- Pelabelan dilakukan dengan **IndoBERT** (`mdhugol/indonesia-bert-sentiment-classification`),
  bukan lexicon-based, sesuai wajib brief.
- Input model menggunakan **teks natural yang dibersihkan ringan** (bukan hasil stem/stopword
  removal Tahap 5) agar konteks kalimat tetap terjaga untuk model kontekstual seperti BERT.
- Mapping label diambil dinamis dari `model.config.id2label` + sanity check kalimat contoh
  sebelum full-run, untuk menghindari kesalahan mapping index yang fatal.
- Proses inferensi dilakukan **per-batch dengan checkpointing** agar aman dari gangguan sesi
  Colab pada dataset besar (118rb+ baris).
- Confidence score disimpan per baris untuk analisis kualitas label lebih lanjut (baris dengan
  confidence rendah bisa jadi kandidat audit manual).